# Flow compensation on a spiral, without a spiral kernel

**What this notebook is for.** SeqCraft ships a packaged kernel for the Cartesian gradient echo,
`GRE2DTR`, and [`gre_2d/03_flow_comp`](../gre_2d/03_flow_comp.ipynb) asks it for flow compensation
with one keyword. There is no packaged spiral kernel — [`01_build`](01_build.ipynb) composes a
spiral repetition out of `Excitation`, `SpiralReadout` and a spoiler, in ordinary user code. This
notebook shows that such a composition can reach the **same** physical designer, without being
promoted into a kernel class first.

> **Provisional internal interface.** The substrate used here, `seqcraft.design.scope`, is
> internal and will change. The public way to declare a physical-design scope for your own
> composition has not been designed yet. Everything about the *physics* below is stable; the
> spelling is not.

## The problem

A spiral-in trajectory starts at the edge of k-space and ends at the origin, so the arm plays in
full **before** the echo and arrives carrying both moments:

$$\phi = 2\pi\left(m_0 x_0 + m_1 v\right)$$

The prephaser that takes k out to the edge has to cancel $m_0$ at the echo — that is what puts
$k = 0$ there. Nothing makes it cancel $m_1$, so a spin moving in the slice plane arrives with a
velocity-dependent phase, exactly as on a Cartesian readout.

What makes this different from the Cartesian case, and the reason it is worth showing, is the
**state**. Each interleaf is the same trajectory rotated, so the moment the prephaser must cancel
rotates with it, on both in-plane axes at once:

```text
angle     m0_x      m0_y          the arm alone, measured to its own echo
0.000  -145.38      0.00
0.785  -102.80   -102.80
1.571    -0.00   -145.38
```

A Cartesian phase encode varies one axis linearly and leaves the other alone. This varies both,
and no packaged kernel in SeqCraft produces it.

| | |
|---|---|
| **1** | the composition, and what it needs |
| **2** | declaring the scope |
| **3** | designing once, realising every interleaf |
| **4** | measuring the emitted repetitions |
| **5** | the file |

**Needs nothing but `seqcraft`.**

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pypulseq as pp

import seqcraft as sc
from seqcraft.design import joint, scope        # internal, provisional

opts = pp.Opts(
    max_grad=38, grad_unit='mT/m',
    max_slew=140, slew_unit='T/m/s',
    rf_dead_time=100e-6,
    rf_ringdown_time=30e-6,
    adc_dead_time=10e-6,
    adc_samples_limit=8192,
)

FOV_MM, MATRIX, THICKNESS_MM = 240.0, 64, 5.0
SHOTS, FLIP_DEG, TR_S = 8, 15.0, 30e-3

SEQ_DIR = Path('seq')
SEQ_DIR.mkdir(exist_ok=True)
raster = sc.Raster(opts.grad_raster_time)

---

## 1. The composition, and what it needs

The same three leaves `01_build` uses, with two differences.

`variant='in'` puts the echo at the end of the arm rather than 12 us into it, which is what gives
the first moment something to be. And `prephase=False` is `SpiralReadout`'s existing way of saying
*I state the moment I need at my start, you realise it* — the same split `CartesianLine(prephase=
False)` and `Excitation.build(rephase=False)` offer. That handed-over prephaser is the waveform
the designer gets to shape.

In [ ]:
exc = sc.modules.Excitation(opts=opts, flip_deg=FLIP_DEG, thickness_mm=THICKNESS_MM,
                            duration_s=1e-3)
arm = sc.modules.SpiralReadout(opts=opts, fov_mm=FOV_MM, matrix=MATRIX, shots=SHOTS,
                               dwell_s=4e-6, variant='in')
spoil = sc.modules.spoiler(opts, cycles_per_voxel=4.0, voxel_mm=THICKNESS_MM, axis='z')

angles = tuple(2.0 * np.pi * i / SHOTS for i in range(SHOTS))

print(f'arm            {arm.duration_s * 1e3:.3f} ms, echo at '
      f'{arm.time_to_echo(0) * 1e6:.0f} us into a block that includes its own prephaser')
print(f'its prephaser  {arm.prephaser_duration_s * 1e6:.0f} us, starting k '
      f'{arm.start_k_per_m:.1f} 1/m')
print(f'interleaves    {SHOTS}')

---

## 2. Declaring the scope

Three numbers say where the adjustable region sits between the two instants that matter — the
excitation, and the echo:

```text
0            origin          window_start            +window          +tail
|              |                  |                     |               |
[-- excitation X ------------------[==== prephaser ====]---- spiral arm --X
               ^                                                          ^
        moments measured from here                                 the echo
```

**One subtlety, and it is the one that bites.** `arm.time_to_echo(0)` is measured from a block
that includes the arm's *own* prephaser. This scope has taken that prephaser over, so its duration
has to come back off — otherwise the declared echo lands past the end of the arm, inside the
rewinder, and the design is solved for an instant the sequence does not have.

The rule behind that: **the instants, the fixed contributions and the adjustable region must all
describe the same physical decomposition.**

In [ ]:
geometry = scope.ScopeGeometry(
    origin_s=exc.time_to_center(),
    window_start_s=float(raster.ceil(exc().duration)),
    tail_s=arm.time_to_echo(0) - arm.prephaser_duration_s,
)

print(f'origin        {geometry.origin_s * 1e3:.3f} ms')
print(f'window starts {geometry.window_start_s * 1e3:.3f} ms')
print(f'tail          {geometry.tail_s * 1e3:.3f} ms  '
      f'(the arm up to its origin crossing, without the prephaser)')

Then one requirement per axis: what is wanted for each state, what already plays, and which states
are worth searching over.

`design_states` is a **proposal**, not a proof. The arm is one trajectory rotated, so the
requirement on `x` is a state-independent magnitude times `cos(angle)` and on `y` times
`sin(angle)`; both are bounded by that magnitude and each bound is reached, on `x` at angle 0 and
on `y` at a quarter turn. Those two angles are therefore the ones worth designing against — and
if that reasoning were wrong, the designer would find out, because it realises every interleaf
before accepting a schedule.

In [ ]:
def requirement(axis):
    def fixed(state, schedule):
        """Everything already on this axis between the two instants, as emitted."""
        played = sc.LogicBlock()
        played.add(0.0, exc())
        played.add(schedule.window_start_s + schedule.window_s,
                   arm(angle_rad=float(state), prephase=False, rewind=False, acquire=False))
        return tuple(
            joint.measure_moment(played, order, axis, origin_s=schedule.origin_s,
                                 start_s=schedule.origin_s, end_s=schedule.endpoint_s)
            for order in joint.ORDERS
        )

    aligned = 0.0 if axis == 'x' else np.pi / 2.0
    nearest = min(angles, key=lambda a: abs(abs(np.cos(a - aligned)) - 1.0))
    return scope.AxisRequirement(
        axis=axis,
        states=angles,
        design_states=(nearest,),
        target=lambda _state: (0.0, 0.0),        # k = 0 at the echo, and no first moment either
        fixed=fixed,
    )


requirements = [requirement('x'), requirement('y')]
for r in requirements:
    print(f'axis {r.axis}: {len(r.states)} states, designing against '
          f'{float(r.design_states[0]):.3f} rad')

---

## 3. Designing once, realising every interleaf

The search runs over the proposed states; every interleaf is then realised at the schedule that
comes out. That split is the point — the timing is a property of the **family**, not of each
state, and solving it per interleaf would give eight different echo times for one image.

In [ ]:
design = scope.design_scope(geometry, requirements, opts,
                            min_window_s=arm.prephaser_duration_s)

print(f'window  {design.window_s * 1e6:.0f} us')
print(f'TE      {geometry.echo_at(design.window_s) * 1e3:.3f} ms, the same for every interleaf')
print(f'lobes   ' + ', '.join(
    f'{axis}: {len(design.designs[axis].realisations[angles[0]].events)}' for axis in ('x', 'y')))

A repetition is built **once**, from the finished design. Nothing goes back and edits a block that
already exists: the events the designer produced are placed alongside the leaves, and the result
is handed to the compiler.

In [ ]:
def repetition(state, *, phase_deg=0.0, acquire=True):
    """Excite, play the designed prephaser, play the interleaf, spoil."""
    out = sc.LogicBlock('spiral_tr')
    out.add(0.0, exc(phase_deg=phase_deg))
    for axis in ('x', 'y'):
        for at, event in design.events_for(axis, state):
            out.add(at, event)
    start = design.schedule.window_start_s + design.window_s
    out.add(start, arm(angle_rad=float(state), prephase=False, acquire=acquire,
                       phase_deg=phase_deg))
    out.add(start + arm(angle_rad=float(state), prephase=False).duration, spoil)
    return out


print(f'one repetition: {repetition(angles[0]).duration * 1e3:.3f} ms')

---

## 4. Measuring the emitted repetitions

Every interleaf, both in-plane axes, both moments, integrated over everything the repetition plays
between the excitation and the echo.

In [ ]:
end_s = geometry.schedule_at(design.window_s).endpoint_s


def measured(state, order, axis):
    return joint.measure_moment(repetition(state), order, axis,
                                origin_s=geometry.origin_s, start_s=geometry.origin_s,
                                end_s=end_s)


print(f'{"angle":>8}{"m0_x / (1/m)":>16}{"m0_y / (1/m)":>16}'
      f'{"m1_x / (s/m)":>16}{"m1_y / (s/m)":>16}')
for state in angles:
    print(f'{state:8.3f}' + ''.join(f'{measured(state, o, a):16.2e}'
                                    for o in (0, 1) for a in ('x', 'y')))

`k` arrives at the origin and the first moment is gone, for every interleaf — and the schedule
they were realised at is one schedule, so they share an echo time.

What that buys physically: a spin moving in the slice plane reaches the echo with the phase it
would have had standing still, whichever interleaf is being played. Without it, that phase would
rotate from shot to shot with the trajectory, which is a shot-to-shot inconsistency in a
reconstruction that assumes all interleaves saw the same object.

**Image-level artefact reduction is not demonstrated here.** What is demonstrated is that the
velocity-dependent phase term goes to zero on the emitted waveform.

---

## 5. The file

In [ ]:
scan = sc.LogicBlock('gre_spiral_2d_flow_comp')
for index, state in enumerate(angles):
    scan.add(index * TR_S, repetition(state, phase_deg=0.5 * 117.0 * index * (index + 1)))

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', sc.SeqCraftWarning)
    seq = sc.compile(scan, opts, name='gre_spiral_2d_flow_comp')

path = SEQ_DIR / 'gre_spiral_2d_flow_comp.seq'
seq.write(str(path))
print(f'{len(seq.block_events)} blocks, {seq.duration()[0]:.3f} s   {path}')
for warning in caught:
    print(f'\n{str(warning.message).split(":")[0]}')

---

## Summary

A spiral-in arm reaches the echo carrying a first moment, and the prephaser that puts $k = 0$
there does nothing about it. Reshaping that prephaser nulls both, and because each interleaf is
the same trajectory rotated, the requirement rotates with the state on both in-plane axes at once.

Measured on the emitted repetitions, $m_0$ and $m_1$ are at the arithmetic floor for all eight
interleaves, realised at one common echo time.

The point of the exercise is where the design happened. This repetition has no kernel class — it
is `Excitation`, `SpiralReadout` and a spoiler composed in a notebook — and it used the same
physical designer that `GRE2DTR` uses for its own winder. A composition does not have to become a
packaged module to get physical design.

**What this notebook does not cover.** The declaration above is a provisional internal interface,
not a public API. The scope carries one adjustable window, so a repetition wanting to design its
prephaser and its rewinder together does not fit it. What plays between the window and the echo
must be the same duration for every state, which is true of a rotated arm and not of a
variable-length one. And only $m_0$ and $m_1$ are represented.

## References

Bernstein, King and Zhou, *Handbook of MRI Pulse Sequences*, Elsevier 2004 — §9.2 for gradient
moment nulling, §17.6 for spiral trajectories and their gradient design.

Nishimura, Irarrazabal and Meyer, *A velocity k-space analysis of flow effects in echo-planar and
spiral imaging*, Magn Reson Med 33(4):549-556, 1995 — why the moment structure of a spiral differs
from a Cartesian readout, and what it does to flowing spins.